In [0]:
import time
start_time = time.time()
print(f"⏱️  Broadcast-optimized execution started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# Load all silver tables as batch DataFrames
order_items = spark.table("second_data_engineering_project.silver.order_items")
orders = spark.table("second_data_engineering_project.silver.orders")
customers = spark.table("second_data_engineering_project.silver.customers")
products = spark.table("second_data_engineering_project.silver.products")
sellers = spark.table("second_data_engineering_project.silver.sellers")
reviews = spark.table("second_data_engineering_project.silver.reviews")
category_translation = spark.table("second_data_engineering_project.silver.product_category_name_translation")
geolocation = spark.table("second_data_engineering_project.silver.geolocation")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Aggregate payments to order level (one order can have multiple payments)
payments = spark.table("second_data_engineering_project.silver.payments")

payments_agg = (
    payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_payment_value"),
        F.count("payment_sequential").alias("payment_count"),
        F.max("payment_installments").alias("max_installments"),
        # Get the primary payment type (the one with highest value)
        F.first("payment_type").alias("primary_payment_type")
    )
)

print(f"Payments aggregated. Total orders with payments: {payments_agg.count()}")

In [0]:
# Deduplicate geolocation by zip code (keep first record per zip)
customer_geo = (
    geolocation
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.first("geolocation_lat").alias("customer_lat"),
        F.first("geolocation_lng").alias("customer_lng")
    )
)

In [0]:
# Deduplicate geolocation by zip code for sellers
seller_geo = (
    geolocation
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.first("geolocation_lat").alias("seller_lat"),
        F.first("geolocation_lng").alias("seller_lng")
    )
)

In [0]:
# Start with order_items as the base fact table (grain: order_id + order_item_id)
master = order_items

# Join orders dimension
master = master.join(orders, on="order_id", how="inner")

# Join customers dimension
master = master.join(customers, on="customer_id", how="left")

# Join customer geolocation (BROADCAST - small aggregated table)
master = master.join(
    F.broadcast(customer_geo),
    master.customer_zip_code_prefix == customer_geo.geolocation_zip_code_prefix,
    how="left"
).drop(customer_geo.geolocation_zip_code_prefix)

In [0]:
# Join products dimension (BROADCAST - 33K records)
master = master.join(F.broadcast(products), on="product_id", how="left")

# Join category translation for English category names (BROADCAST - tiny lookup table)
master = master.join(
    F.broadcast(category_translation),
    on="product_category_name",
    how="left"
)

# Join sellers dimension (BROADCAST - 3K records)
master = master.join(F.broadcast(sellers), on="seller_id", how="left")

# Join seller geolocation (BROADCAST - small aggregated table)
master = master.join(
    F.broadcast(seller_geo),
    master.seller_zip_code_prefix == seller_geo.geolocation_zip_code_prefix,
    how="left"
).drop(seller_geo.geolocation_zip_code_prefix)

In [0]:
# Join aggregated payments (BROADCAST - aggregated to order level)
master = master.join(F.broadcast(payments_agg), on="order_id", how="left")

# Join reviews (left join since not all orders have reviews)
# Note: Reviews table is ~100K records, broadcasting may help
reviews_subset = reviews.select(
    "order_id",
    "review_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
)
master = master.join(F.broadcast(reviews_subset), on="order_id", how="left")

In [0]:
# Add calculated metrics for business analytics
master = (
    master
    # Order-level metrics
    .withColumn("order_total_value", F.col("price") + F.col("freight_value"))
    .withColumn(
        "delivery_days",
        F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp"))
    )
    .withColumn(
        "estimated_delivery_days",
        F.datediff(F.col("order_estimated_delivery_date"), F.col("order_purchase_timestamp"))
    )
    .withColumn(
        "delivery_performance",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff(F.col("order_estimated_delivery_date"), F.col("order_delivered_customer_date"))
        ).otherwise(None)
    )
    .withColumn(
        "is_late_delivery",
        F.when(
            F.col("delivery_performance") < 0,
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    # Date dimensions
    .withColumn("order_year", F.year("order_purchase_timestamp"))
    .withColumn("order_month", F.month("order_purchase_timestamp"))
    .withColumn("order_quarter", F.quarter("order_purchase_timestamp"))
    .withColumn("order_day_of_week", F.dayofweek("order_purchase_timestamp"))
    .withColumn("order_date", F.to_date("order_purchase_timestamp"))
    # Product volume (in cubic cm)
    .withColumn(
        "product_volume_cm3",
        F.col("product_length_cm") * F.col("product_width_cm") * F.col("product_height_cm")
    )
    # Review flag
    .withColumn("has_review", F.when(F.col("review_id").isNotNull(), F.lit(True)).otherwise(F.lit(False)))
)

In [0]:
# Select and organize columns in logical order
master_final = master.select(
    # Primary Keys
    "order_id",
    "order_item_id",
    
    # Order Information
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "order_date",
    "order_year",
    "order_month",
    "order_quarter",
    "order_day_of_week",
    
    # Customer Information
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    "customer_lat",
    "customer_lng",
    
    # Product Information
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_volume_cm3",
    
    # Seller Information
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix",
    "seller_lat",
    "seller_lng",
    
    # Order Item Details
    "price",
    "freight_value",
    "order_total_value",
    "shipping_limit_date",
    
    # Payment Information
    "total_payment_value",
    "payment_count",
    "primary_payment_type",
    "max_installments",
    
    # Review Information
    "review_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp",
    "has_review",
    
    # Calculated Metrics
    "delivery_days",
    "estimated_delivery_days",
    "delivery_performance",
    "is_late_delivery"
)

In [0]:
# Display sample records
display(master_final.limit(10))

In [0]:
# Write master table to gold layer
# Partition by year and month for query performance
master_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_year", "order_month") \
    .option("overwriteSchema", "true") \
    .saveAsTable("second_data_engineering_project.gold.orders_master")

end_time = time.time()
execution_time = end_time - start_time
print(f"✅ Master table written to: second_data_engineering_project.gold.orders_master")
print(f"⏱️  Total execution time: {execution_time:.2f} seconds ({execution_time/60:.2f} minutes)")